<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Bubble Plots**


Estimated time needed: **30** minutes


In this lab, you will focus on visualizing data.

The dataset will be directly loaded into pandas for analysis and visualization.

You will use various visualization techniques to explore the data and uncover key trends.


## Objectives


In this lab, you will perform the following:


-   Visualize the distribution of data.

-   Visualize the relationship between two data features.

-   Visualize composition of data.

-   Visualize comparison of data.


#### Setup: Working with the Database
**Install and import the needed libraries**


In [ ]:
!pip install pandas 
!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt

**Download and connect to the database file containing survey data.**


To start, download and load the dataset into a `pandas` DataFrame.



In [ ]:
# Step 1: Download the dataset
!wget -O survey-data.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

# Load the data
df = pd.read_csv("survey-data.csv")

# Display the first few rows of the data to understand its structure
df.head()


### Task 1: Exploring Data Distributions Using Bubble Plots


#### 1. Bubble Plot for Age vs. Frequency of Participation


- Visualize the relationship between respondents’ age and their participation frequency (`SOPartFreq`) using a bubble plot.

- Use the size of the bubbles to represent their job satisfaction (`JobSat`).


In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

df['Age_numeric'] = df['Age'].map(AGE_MAP)
df['JobSat'] = pd.to_numeric(df['JobSat'], errors='coerce')

def bubble_by_group(multi_col, group_col, top_items=10, top_groups=None, title=''):
    """Bubble grid: x = group, y = item, bubble size = how often the item appears in that group."""
    d = df[[multi_col, group_col]].dropna()
    if top_groups:
        d = d[d[group_col].isin(d[group_col].value_counts().head(top_groups).index)]
    d = d.assign(Item=d[multi_col].str.split(';')).explode('Item')
    top = d['Item'].value_counts().head(top_items).index
    counts = d[d['Item'].isin(top)].groupby([group_col, 'Item']).size().reset_index(name='Count')
    if group_col == 'Age':
        counts = counts[counts[group_col].isin(AGE_ORDER)]
        counts[group_col] = pd.Categorical(counts[group_col], categories=AGE_ORDER, ordered=True)
        counts = counts.sort_values(group_col)
        counts[group_col] = counts[group_col].astype(str)
    plt.figure(figsize=(13, 7))
    plt.scatter(counts[group_col], counts['Item'], s=counts['Count'] / counts['Count'].max() * 2000,
                alpha=0.6, color='steelblue', edgecolors='black')
    plt.title(title)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

def bubble_vs_sat(multi_col, top_items=10, title=''):
    """Bubble chart: x = median JobSat, y = item, bubble size = number of respondents."""
    d = df[[multi_col, 'JobSat']].dropna()
    d = d.assign(Item=d[multi_col].str.split(';')).explode('Item')
    s = d.groupby('Item').agg(Respondents=('JobSat', 'size'), Median_JobSat=('JobSat', 'median'))
    s = s.sort_values('Respondents', ascending=False).head(top_items)
    plt.figure(figsize=(11, 7))
    plt.scatter(s['Median_JobSat'], s.index, s=s['Respondents'] / s['Respondents'].max() * 2000,
                alpha=0.6, color='darkorange', edgecolors='black')
    plt.title(title)
    plt.xlabel('Median JobSat (0-10)')
    plt.tight_layout()
    plt.show()
    print(s)

# ---- 1.1 Age vs participation frequency (SOPartFreq), bubble size = job satisfaction ----
d = df[['Age', 'SOPartFreq', 'JobSat']].dropna()
g = d.groupby(['Age', 'SOPartFreq']).agg(Respondents=('JobSat', 'size'),
                                         Mean_JobSat=('JobSat', 'mean')).reset_index()
g = g[g['Age'].isin(AGE_ORDER)]   # drop 'Prefer not to say'
g['Age'] = pd.Categorical(g['Age'], categories=AGE_ORDER, ordered=True)
g = g.sort_values('Age')
g['Age'] = g['Age'].astype(str)

plt.figure(figsize=(13, 7))
plt.scatter(g['Age'], g['SOPartFreq'], s=g['Mean_JobSat'] * 60, c=g['Respondents'],
            cmap='viridis', alpha=0.7, edgecolors='black')
plt.colorbar(label='Number of respondents')
plt.title('Age vs Stack Overflow Participation Frequency (bubble size = average JobSat)')
plt.xlabel('Age group')
plt.ylabel('SOPartFreq')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

#### 2. Bubble Plot for Compensation vs. Job Satisfaction


-Visualize the relationship between yearly compensation (`ConvertedCompYearly`) and job satisfaction (`JobSat`).

- Use the size of the bubbles to represent respondents’ age.


In [ ]:
# ---- 1.2 Compensation vs job satisfaction, bubble size = age ----
d = df[['ConvertedCompYearly', 'JobSat', 'Age_numeric']].dropna()
d = d[d['ConvertedCompYearly'] <= d['ConvertedCompYearly'].quantile(0.99)]

plt.figure(figsize=(12, 7))
plt.scatter(d['ConvertedCompYearly'], d['JobSat'] + np.random.uniform(-0.3, 0.3, len(d)),
            s=d['Age_numeric'] * 2, alpha=0.3, color='seagreen', edgecolors='w', linewidth=0.5)
plt.title('Compensation vs Job Satisfaction (bubble size = age)')
plt.xlabel('ConvertedCompYearly (USD)')
plt.ylabel('JobSat (0-10)')
plt.show()

### Task 2: Analyzing Relationships Using Bubble Plots


#### 1. Bubble Plot of Technology Preferences by Age

- Visualize the popularity of programming languages respondents have worked with (`LanguageHaveWorkedWith`) across age groups.

- Use bubble size to represent the frequency of each language.



In [ ]:
# ---- 2.1 Languages worked with across age groups ----
bubble_by_group('LanguageHaveWorkedWith', 'Age', top_items=10,
                title='Top 10 Languages by Age Group (bubble size = number of users)')

#### 2. Bubble Plot for Preferred Databases vs. Job Satisfaction

- Explore the relationship between preferred databases (`DatabaseWantToWorkWith`) and job satisfaction.

- Use bubble size to indicate the number of respondents for each database.


In [ ]:
# ---- 2.2 Preferred databases vs job satisfaction ----
bubble_vs_sat('DatabaseWantToWorkWith', top_items=10,
              title='Preferred Databases vs Median Job Satisfaction (bubble size = respondents)')

### Task 3: Comparing Data Using Bubble Plots


#### 1. Bubble Plot for Compensation Across Developer Roles

- Visualize compensation (`ConvertedCompYearly`) across different developer roles (`DevType`).

- Use bubble size to represent job satisfaction.


In [ ]:
# ---- 3.1 Compensation across developer roles, bubble size = job satisfaction ----
d = df[['DevType', 'ConvertedCompYearly', 'JobSat']].dropna()
top_roles = d['DevType'].value_counts().head(10).index
s = d[d['DevType'].isin(top_roles)].groupby('DevType').agg(
    Median_Comp=('ConvertedCompYearly', 'median'), Mean_JobSat=('JobSat', 'mean'))
s = s.sort_values('Median_Comp')

plt.figure(figsize=(12, 7))
plt.scatter(s['Median_Comp'], s.index, s=s['Mean_JobSat'] * 150, alpha=0.6,
            color='purple', edgecolors='black')
plt.title('Median Compensation by Developer Role (bubble size = average JobSat)')
plt.xlabel('Median ConvertedCompYearly (USD)')
plt.tight_layout()
plt.show()
print(s)

#### 2. Bubble Plot for Collaboration Tools by Age

- Visualize the relationship between the collaboration tools used (`NEWCollabToolsHaveWorkedWith`) and age groups.

- Use bubble size to represent the frequency of tool usage.


In [ ]:
# ---- 3.2 Collaboration tools by age group ----
bubble_by_group('NEWCollabToolsHaveWorkedWith', 'Age', top_items=10,
                title='Top 10 Collaboration Tools by Age Group (bubble size = number of users)')

### Task 4: Visualizing Technology Trends Using Bubble Plots


#### 1. Bubble Plot for Preferred Web Frameworks vs. Job Satisfaction

- Explore the relationship between preferred web frameworks (`WebframeWantToWorkWith`) and job satisfaction.

- Use bubble size to represent the number of respondents.



In [ ]:
# ---- 4.1 Preferred web frameworks vs job satisfaction ----
bubble_vs_sat('WebframeWantToWorkWith', top_items=10,
              title='Preferred Web Frameworks vs Median Job Satisfaction (bubble size = respondents)')

#### 2. Bubble Plot for Admired Technologies Across Countries

- Visualize the distribution of admired technologies (`LanguageAdmired`) across different countries (`Country`).

- Use bubble size to represent the frequency of admiration.



In [ ]:
# ---- 4.2 Admired languages across countries (top 10 countries) ----
bubble_by_group('LanguageAdmired', 'Country', top_items=10, top_groups=10,
                title='Top 10 Admired Languages Across the Top 10 Countries (bubble size = admirers)')

## Final Step: Review


After completing the lab, you will have extensively used bubble plots to gain insights into developer community preferences, demographics, compensation trends, and job satisfaction.


## Summary


After completing this lab, you will be able to:

- Create and interpret bubble plots to analyze relationships and compositions within datasets.

- Use bubble plots to explore developer preferences, compensation trends, and satisfaction levels.

- Apply bubble plots to visualize complex relationships involving multiple dimensions effectively.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2024-10-29|1.2|Madhusudhan Moole|Updated lab|
|2024-10-16|1.1|Madhusudhan Moole|Updated lab|
|2024-10-15|1.0|Raghul Ramesh|Created lab|
--!>


Copyright © IBM Corporation. All rights reserved.
